# 01. Load, merge, QC and sample metadata

Reanalysis of GSE217930 (Dooley et al. 2023, PBMC scRNA-seq before and after malaria treatment).

This notebook reads the 20 Cell Ranger libraries, merges them, applies QC, attaches donor / day / batch from GEO and
drops the failed Child1 day 0 library. Output: `qc_annotated.h5ad` (raw counts) in `out_dir`.

Download `GSE217930_RAW.tar` from GEO and extract it into `data_dir` first.

In [ ]:
from pathlib import Path
from importlib.metadata import version
import numpy as np
import pandas as pd
import scanpy as sc
import anndata as ad
import matplotlib.pyplot as plt

# ---- edit these two paths ----------------------------------------------------
data_dir = Path("data")                        # folder with the extracted GSE217930 files
out_dir = Path("E:/malaria_scrna_results")     # large intermediate files, needs about 5 GB free
# ------------------------------------------------------------------------------
fig_dir = out_dir / "figures"
tab_dir = out_dir / "tables"
for d in (out_dir, fig_dir, tab_dir):
    d.mkdir(parents=True, exist_ok=True)
sc.settings.figdir = fig_dir
sc.set_figure_params(dpi=100, dpi_save=200, facecolor="white")
print({p: version(p) for p in ['scanpy', 'anndata', 'pandas', 'numpy']})


## Load and merge the 20 libraries
Libraries come from two sequencing batches with different feature lists, so genes are matched by Ensembl ID and only genes present in all libraries are kept (inner join).

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import scanpy as sc
import anndata as ad

merged_path = out_dir / "merged_raw_v2.h5ad"

if merged_path.exists():
    adata = sc.read_h5ad(merged_path)
    print("loaded saved file:", adata)
else:
    prefixes = sorted(p.name.replace("matrix.mtx.gz", "") for p in data_dir.glob("*_matrix.mtx.gz"))
    print(len(prefixes), "samples found")

    adatas = {}
    for prefix in prefixes:
        a = sc.read_10x_mtx(data_dir, prefix=prefix, cache=False)
        a.var["gene_symbol"] = a.var_names.values
        a.var_names = a.var["gene_ids"].astype(str).values
        a.var_names_make_unique()
        a.obs["gsm"] = prefix.split("_")[0]
        a.obs["sample_code"] = prefix.split("_")[1]
        adatas[prefix.split("_")[1]] = a
        print(prefix, a.shape)

    adata = ad.concat(adatas, join="inner", label="sample_key", index_unique="-")
    symbols = pd.concat([a.var["gene_symbol"] for a in adatas.values()]).groupby(level=0).first()
    adata.var["gene_symbol"] = symbols.reindex(adata.var_names).values
    adata.write_h5ad(merged_path, compression="gzip")
    print("built and saved:", adata)


In [ ]:
print(adata)

## Quality control
Thresholds follow the paper: at least 250 genes, at least 500 UMIs, under 20% mitochondrial reads, complexity above 0.8. Haemoglobin genes are removed, then genes seen in fewer than 10 cells.

In [ ]:
#Quality control
import numpy as np
import pandas as pd
import scanpy as sc

adata.var["mt"] = adata.var["gene_symbol"].astype(str).str.startswith("MT-")
hb_genes = ["HBA1","HBA2","HBB","HBD","HBE1","HBG1","HBG2","HBM","HBQ1","HBZ"]
adata.var["hb"] = adata.var["gene_symbol"].isin(hb_genes)

sc.pp.calculate_qc_metrics(adata, qc_vars=["mt","hb"], percent_top=None, log1p=False, inplace=True)
adata.obs["complexity"] = np.log10(adata.obs["n_genes_by_counts"]) / np.log10(adata.obs["total_counts"])

sc.pl.violin(adata, ["n_genes_by_counts","total_counts","pct_counts_mt"],
             groupby="sample_code", rotation=90, multi_panel=True)

keep = ((adata.obs["n_genes_by_counts"] >= 250) & (adata.obs["total_counts"] >= 500) &
        (adata.obs["pct_counts_mt"] < 20) & (adata.obs["complexity"] > 0.8))
before = adata.obs["sample_code"].value_counts().sort_index()
after = adata.obs.loc[keep, "sample_code"].value_counts().sort_index()
print(pd.DataFrame({"before": before, "after": after}).to_string())
print("cells kept:", keep.sum(), "of", len(keep))

adata = adata[keep, ~adata.var["hb"]].copy()
sc.pp.filter_genes(adata, min_cells=10)
adata.layers["counts"] = adata.X.copy()
adata.write_h5ad(out_dir / "qc.h5ad", compression="gzip")
print(adata)


## Sample metadata from GEO
Donor, day, age group and sequencing batch come from the GEO sample titles. Needs an internet connection the first time.

In [ ]:
import GEOparse
gse = GEOparse.get_GEO("GSE217930", destdir=str(out_dir), silent=True)

rows = []
for gsm_name, gsm in gse.gsms.items():
    rows.append({"gsm": gsm_name,
                 "title": gsm.metadata.get("title", [""])[0],
                 "characteristics": " | ".join(gsm.metadata.get("characteristics_ch1", []))})
meta = pd.DataFrame(rows).sort_values("gsm")
print(meta.to_string())
meta.to_csv(out_dir / "geo_sample_metadata.csv", index=False)


In [ ]:
meta = pd.read_csv(out_dir / "geo_sample_metadata.csv")
meta["donor"] = meta["title"].str.split("_").str[0]
meta["day"] = meta["title"].str.extract(r"Day(\d+)")[0].fillna("EC")
meta["group"] = np.where(meta["donor"].str.startswith("Control"), "control", "infected")
meta["age"] = meta["characteristics"].str.extract(r"age: (\d+)")[0].astype(int)
meta["age_group"] = np.where(meta["donor"].str.startswith("Child"), "child", "adult")
meta["batch"] = np.where(meta["gsm"] <= "GSM6730086", "batch1", "batch2")

m = meta.set_index("gsm")
for col in ["title", "donor", "day", "group", "age", "age_group", "batch"]:
    adata.obs[col] = adata.obs["gsm"].astype(str).map(m[col]).values

print("unmapped cells:", adata.obs["donor"].isna().sum())

# drop the failed sample, as in the paper
adata = adata[adata.obs["title"] != "Child1_Day0"].copy()

print(pd.crosstab(adata.obs["donor"], adata.obs["day"]))
print(pd.crosstab(adata.obs["batch"], adata.obs["day"]))
print(adata)
adata.write_h5ad(out_dir / "qc_annotated.h5ad", compression="gzip")
print("saved qc_annotated.h5ad")
